# 공고돋보기 — 규칙 검토 노트북 (rule_review, 학습 고리)
저장된 사례(동의 사례·평가용 가상 사례) → 보관 기한 지난 사례 삭제 → **AI가 묶고 규칙 후보 초안** → **사람이 공식 근거를 확인해 승인** → 회귀 시험 → `rules.yaml` 변경안(diff).

- 사용자 피드백·AI 초안만으로는 규칙이 바뀌지 않는다(사람 승인 + 근거 ID 필수).
- 승인된 규칙도 앱에 바로 반영하지 않고 변경안 파일로 낸다. 대회 제출 후에는 규칙을 바꾸지 않는다.
- 사람의 결정은 `notebooks/review_decisions.json` 에 적는다(승인자·근거 ID).

In [1]:
# 환경 준비: Colab 이면 배포용 공개 저장소(app/ 만 담은 저장소)를 받아 오고, 키는 Colab '보안 비밀'에서 읽는다.
import os, sys, json, pathlib, warnings
warnings.filterwarnings("ignore")  # 라이브러리 경고(로컬 경로가 찍힘)를 실행 기록에 남기지 않는다
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    APP_REPO_URL = "https://github.com/jmjmoon02-prog/gonggo-dotbogi.git"  # 배포용 저장소(app/ 만, 제출 시 공개)
    if not pathlib.Path("app").exists():
        !git clone -q $APP_REPO_URL app
    !pip install -q -r app/requirements.txt
    from google.colab import userdata
    for k in ["OPENAI_API_KEY", "MODEL_ID", "HF_TOKEN", "HF_DATASET_REPO"]:
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
    APP_DIR = pathlib.Path("app").resolve()
else:
    APP_DIR = pathlib.Path.cwd().resolve().parent  # notebooks/ 의 상위 = app/
sys.path.insert(0, str(APP_DIR))
RESULTS = APP_DIR / "notebooks" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
import pandas as pd
pd.set_option("display.max_colwidth", 80)
from core import config
print("app:", APP_DIR.name, "| model:", config.model_id(), "| OpenAI 키:", "있음" if config.get_secret("OPENAI_API_KEY") else "없음")

app: app | model: gpt-6.2-sol | OpenAI 키: 있음


## 1. 사례 불러오기 · 보관 기한 지난 사례 삭제

In [2]:
import tempfile
from huggingface_hub import HfApi
from core import review, storage
from core import evaluation as ev
from core.rules_engine import load_rules
RB = load_rules()
token, repo = config.get_secret("HF_TOKEN"), config.get_secret("HF_DATASET_REPO")
tmp = pathlib.Path(tempfile.mkdtemp())
cases = review.load_cases_from_hf(token, repo, tmp)
expired = review.purge_expired(cases, api=HfApi(), token=token, repo_id=repo)
cases = [c for c in cases if c["_path"] not in set(expired)]
print(f"사례 {len(cases)}건 (기한 지나 삭제 {len(expired)}건)")
display(pd.DataFrame([{"case_id": c["case_id"], "계기": c.get("trigger_label", c.get("trigger")), "가상": c.get("synthetic", False),
                       "등급": c["verdict"]["grade"], "신종 의심": c["verdict"]["novel_suspect"], "걸린 규칙": ",".join(c["verdict"]["matched_ids"]),
                       "기한": c["expires_at"]} for c in cases]))

사례 22건 (기한 지나 삭제 0건)


,case_id,계기,가상,등급,신종 의심,걸린 규칙,기한
0,0b1b8314b038,평가용 가상 사례(검증 노트북의 '유형 하나 빼기'에서 신종 의심으로 판정된 테스트셋 항목),True,red,True,AX_A,2027-02-21
1,0e2931d30ddd,평가용 가상 사례(검증 노트북의 '유형 하나 빼기'에서 신종 의심으로 판정된 테스트셋 항목),True,red,True,"AX_A,S3",2027-02-21
2,25c62a76d78f,평가용 가상 사례(검증 노트북의 '유형 하나 빼기'에서 신종 의심으로 판정된 테스트셋 항목),True,red,True,"AX_A,AX_F,S2",2027-02-21
3,2bcbcfae279e,평가용 가상 사례(검증 노트북의 '유형 하나 빼기'에서 신종 의심으로 판정된 테스트셋 항목),True,red,True,"AX_A,S2",2027-02-21
4,30f7ea0170d3,평가용 가상 사례(검증 노트북의 '유형 하나 빼기'에서 신종 의심으로 판정된 테스트셋 항목),True,red,True,"AX_D,S1,S3",2027-02-21
5,34482d4c32c0,평가용 가상 사례(검증 노트북의 '유형 하나 빼기'에서 신종 의심으로 판정된 테스트셋 항목),True,red,True,"AX_A,AX_F",2027-02-21
6,39488851b47a,평가용 가상 사례(검증 노트북의 '유형 하나 빼기'에서 신종 의심으로 판정된 테스트셋 항목),True,red,True,AX_A,2027-02-21
7,3f6e7754605f,평가용 가상 사례(검증 노트북의 '유형 하나 빼기'에서 신종 의심으로 판정된 테스트셋 항목),True,red,True,AX_A,2027-02-21
8,46601f9da6da,평가용 가상 사례(검증 노트북의 '유형 하나 빼기'에서 신종 의심으로 판정된 테스트셋 항목),True,red,True,AX_A,2027-02-21
9,508d5d91895e,평가용 가상 사례(검증 노트북의 '유형 하나 빼기'에서 신종 의심으로 판정된 테스트셋 항목),True,red,True,AX_A,2027-02-21


## 2. AI 묶기 · 규칙 후보 초안 (결과는 캐시)

In [3]:
from core.extract import make_client
draft_path = RESULTS / "review_draft.json"
if draft_path.exists():
    draft = review.ReviewDraft.model_validate_json(draft_path.read_text(encoding="utf-8")); usage = None
    print("캐시된 초안 사용(추가 비용 없음)")
else:
    draft, usage = review.draft_candidates(cases, make_client(), config.model_id(), RB)
    draft_path.write_text(draft.model_dump_json(indent=1), encoding="utf-8")
    from core.extract import estimate_cost_usd
    print("초안 생성 비용 약 $", round(estimate_cost_usd(usage) or 0, 4), usage)
display(pd.DataFrame([c.model_dump() for c in draft.clusters]))
cand_rows = []
for c in draft.candidates:
    checks = review.verify_candidate_examples(c, cases)
    cand_rows.append({"임시 ID": c.suggested_id, "등급": c.tier, "축": c.axis, "이름": c.name, "설명": c.description,
                      "예시(원문 확인)": [f"{e} {'✅' if ok else '❌'}" for e, ok in checks], "확인 질문": c.check_question,
                      "근거 사례": c.case_ids, "이유": c.rationale, "근거 검색어": c.evidence_search_hint})
display(pd.DataFrame(cand_rows))

캐시된 초안 사용(추가 비용 없음)


,label,case_ids,summary
0,채용·가입 단계의 개인정보 및 인증정보 제출,"[0b1b8314b038, 0e2931d30ddd]",일정 안내나 업무 시작 전에 지원자의 신분증 사진·계좌번호 또는 본인인증 번호가 채용 담당자 측으로 넘어간다. 각각 H4와 H1의 기존 ...
1,구매·리뷰·미션을 조건으로 한 선결제,"[25c62a76d78f, 2bcbcfae279e, 39488851b47a, a175dc25f2ad]","참여자가 구매비·충전금·미션 금액을 먼저 부담하고, 이후 원금 환급이나 추가 수익을 약속받는다. 첫 단계가 무료여도 후속 단계에 입금이 ..."
2,배송 일감 배정을 조건으로 한 차량 비용 부담,"[46601f9da6da, a404248dc473, e4513547f932]","배송 업무를 받기 위해 지원자가 지정 차량 구매비나 리스 관련 계약금을 부담한다. 일부는 대출 주선 또는 신용조회 동의 요구를 동반하며,..."
3,개인 명의 회선·계정의 제공 또는 업무 사용,"[34482d4c32c0, 3f6e7754605f, 508d5d91895e, 614fa4685180]","지원자가 본인 명의 유심이나 기존 계정의 접근권을 넘기거나, 본인 명의 유심을 대량 발송 업무에 사용한다. 직접 인도가 없는 대량 발송 ..."
4,제3자 자금의 수령·변환·재전달,"[8a26f83e176a, 935d0861778f, d4e85728b52f]","근로자가 타인의 카드·현금·거래대금을 받아 지정처로 옮긴다. 개인 계좌를 통한 이체뿐 아니라 카드 출금 후 입금, 현금을 상품권 핀번호로..."
5,제3자 물품의 수령·운반·재발송,"[b2eb665f6881, bac2c89aaeba, f826cf40ddfc]","근로자가 물건이나 봉투를 받아 지정 장소 또는 주소로 전달한다. 일부는 내용물 확인을 명시적으로 막으며, H2가 공통 흐름을 포착한다. ..."
6,해외 근무 조건에 따른 여권 통제 상실 또는 취업 자격 미확인,"[30f7ea0170d3, e7150d3ac5ae]","해외 근무와 높은 보수를 제시하지만 위험 조건은 서로 다르다. 한 사례는 출국 후 여권이 회사로 넘어가고, 다른 사례는 근무 전 취업 자..."
7,지원 자격 제한과 고액 보상 제시,[ca7e8e180dd3],"특정 성별·연령에 한해 간단한 업무와 고액 일급을 제시한다. 지원자의 돈·물건·명의가 이전되는 요구는 확인되지 않으며, 성별·연령 제한과..."


,임시 ID,등급,축,이름,설명,예시(원문 확인),확인 질문,근거 사례,이유,근거 검색어
0,C01,H,D,근무 조건으로 여권 원본 보관 요구,구인자나 고용주가 근무 또는 현지 체류 조건으로 여권 원본을 맡기도록 하여 근로자의 직접 소지를 제한한다. 본인 확인을 위한 일시 제시 ...,[출국 후 여권은 회사에서 보관합니다. ✅],"회사가 여권을 확인하고 즉시 돌려주는 것이 아니라, 근무 또는 체류 조건으로 맡겨 두라고 하는가?",[30f7ea0170d3],"해당 공고는 H7에도 해당하지만, H7은 해외 근무와 항공권·숙식·급여 등 유인 조건을 포착한다. 여권 원본 보관은 그런 혜택이 없어도 ...",외교부 해외안전여행 해외취업 사기 여권 압수 인신매매 여권 보관
1,C02,H,C,취업 자격 확보 전 근무 요구,관광·방문 자격 등으로 입국한 뒤 필요한 취업 자격이나 근로허가를 확보하기 전에 근무하도록 안내한다. 입국 후 적법한 자격을 갖춘 다음 ...,[관광비자로 입국 후 근무 ✅],"공고가 관광·방문 자격으로 입국한 뒤 근무하라고 하면서, 근무 시작 전에 필요한 취업 자격이나 허가를 확보하는 절차를 제시하지 않는가?",[e7150d3ac5ae],"H7은 해외 근무에 붙는 유인 조건이나 무비자 안내를 포착하지만, 비자가 있더라도 해당 자격으로 근무할 수 있는지는 별개다. 급여나 숙식...",주캄보디아 대한민국 대사관 캄보디아 관광비자 취업 근로허가 외국인 work permit 공식 안내


## 3. 사람 승인 (`review_decisions.json`)

In [4]:
decisions_path = APP_DIR / "notebooks" / "review_decisions.json"
decisions = json.loads(decisions_path.read_text(encoding="utf-8")) if decisions_path.exists() else {"approvals": []}
print(json.dumps(decisions, ensure_ascii=False, indent=1))

{
 "reviewed_at": "2026-11-23",
 "approver_note": "승인자는 참가자 본인(CrAI LAB). 채팅으로 결정을 받아 기록함(사용자 답: 'C01 승인, C02 보류').",
 "approvals": [
  {
   "suggested_id": "C01",
   "rule_id": "H9",
   "evidence_ids": [
    "E60"
   ],
   "approver": "참가자 본인(CrAI LAB)",
   "status": "candidate",
   "reason": "E60(KTV·정책브리핑 2025-11-19)이 해외 취업사기에서 '여권·휴대전화 압수 후 감금'을 직접 다룬다. 후보(candidate) 상태로만 기록하고 판정에는 반영하지 않는다."
  }
 ],
 "rejections": [
  {
   "suggested_id": "C02",
   "reason": "E60은 '무비자 취업은 현지 불법체류 처벌 가능'만 다룬다. '관광비자로 입국 후 근무'를 직접 뒷받침하는 공식 근거가 없어 보류. 해당 시나리오(G6-3)는 기존 H7로 이미 🔴 판정됨."
  }
 ]
}


## 4. 변경안 만들기 · 회귀 시험 (테스트셋 추출 재사용, 무비용)

In [5]:
import yaml
rules_text = config.RULES_PATH.read_text(encoding="utf-8")
new_data, added = review.apply_approvals(yaml.safe_load(rules_text), draft, decisions, cases)
print("승인되어 변경안에 들어간 규칙:", [r["id"] for r in added] or "없음")
items = ev.load_items()
eval_cache = json.loads((RESULTS / "eval_cache_testset.json").read_text(encoding="utf-8"))
reg = review.regression(new_data, items, eval_cache)
print(json.dumps(reg, ensure_ascii=False, indent=1))
diff = review.rules_yaml_diff(rules_text, added)
(RESULTS / "rules_candidate.diff").write_text(diff or "# 승인된 후보 없음\n", encoding="utf-8")
summary = {"run_at": pd.Timestamp.now(tz="Asia/Seoul").isoformat(timespec="seconds"), "cases": len(cases), "expired_deleted": len(expired),
           "clusters": len(draft.clusters), "candidates": [c.suggested_id + " " + c.name for c in draft.candidates],
           "approved": [r["id"] for r in added], "regression": reg}
(RESULTS / "review_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print(diff or "변경안 없음")

승인되어 변경안에 들어간 규칙: ['H9']
{
 "ok": true,
 "errors": [],
 "changes": [],
 "before": {
  "탐지율": 1.0,
  "오탐률": 0.05
 },
 "after": {
  "탐지율": 1.0,
  "오탐률": 0.05
 },
 "note": "오탐률 증가 없음"
}
--- rules/rules.yaml
+++ rules/rules.yaml (변경안)
@@ -490,6 +490,29 @@
     evidence_ids: [E66, MW2026]
     version: "2.0.0"
     status: active
+
+  # ---- 규칙 검토 노트북에서 사람이 승인한 후보 (변경안) ----
+  - id: H9
+    tier: H
+    axis: D
+    detection: ai
+    name: 근무 조건으로 여권 원본 보관 요구
+    description: 구인자나 고용주가 근무 또는 현지 체류 조건으로 여권 원본을 맡기도록 하여 근로자의 직접 소지를 제한한다. 본인 확인을 위한 일시 제시 후 즉시 반환하는 절차는 제외한다.
+    examples:
+    - 출국 후 여권은 회사에서 보관합니다.
+    check_question: 회사가 여권을 확인하고 즉시 돌려주는 것이 아니라, 근무 또는 체류 조건으로 맡겨 두라고 하는가?
+    guidance: null
+    legal_note: null
+    report_to: []
+    evidence_ids:
+    - E60
+    version: 2.2.0-draft
+    status: candidate
+    review:
+      approver: 참가자 본인(CrAI LAB)
+      date: '2026-11-23'
+      source_cases:
+      - 30f7ea0170d3
 
 # 신종 대응 원칙 6축 (H·M 에 안 걸려도 가치 흐름이 원칙을 어기면 판정)
 